# Step 6: does a deep model change anything?

- **Model:** a small GRU in PyTorch. It reads the last 20 days in order, oldest first, and keeps a memory as it goes.
- **Compared with:** logistic regression on exactly the same inputs and days.
- **Tuning:** hidden size and epochs are picked on the tune year, with early stopping, as in step 4.

In [1]:
from functools import partial

import pandas as pd
from joblib import Parallel, delayed

from classifiers import CLASSIFIERS, fit_classifier, model_table, one_answer_folds, overview
from data import load_all, load_us
from evaluation import mcnemar
from features import make_sequence, make_sequence_with_us, make_us_features, up_close_to_close, up_open_to_close
from gru import HIDDEN_SIZES, fit_gru, pick_gru_setting
from models import pick_by_log_loss, pooled, walk_forward

us = load_us()
markets = {market: df[:"2022"] for market, df in load_all().items()}  # the US file ends in 2022

def with_us(df):
    return make_sequence_with_us(df, us)

## Run both models on three tasks

The markets run in parallel. About 20 minutes.

In [2]:
TASKS = {
    "own returns, close-to-close": (make_sequence, up_close_to_close),
    "own returns + US, close-to-close": (with_us, up_close_to_close),
    "own returns + US, open-to-close": (with_us, up_open_to_close),
}

def run_all_markets(fit, settings, pick, make_x, make_y):
    runs = Parallel(n_jobs=6)(
        delayed(walk_forward)(df, fit, settings, make_x=make_x, make_y=make_y, pick=pick)
        for df in markets.values()
    )
    return dict(zip(markets, runs))

gru, lr = {}, {}
for task, (make_x, make_y) in TASKS.items():
    gru[task] = run_all_markets(fit_gru, HIDDEN_SIZES, pick_gru_setting, make_x, make_y)
    lr[task] = run_all_markets(fit_classifier, CLASSIFIERS["Logistic regression"], pick_by_log_loss, make_x, make_y)

## GRU vs logistic regression

All six test years pooled. **p GRU vs LR** is McNemar's test on the days where only one of them was right.

In [3]:
def compare(gru_preds, lr_preds):
    rows = {}
    for market in markets:
        g, l = gru_preds[market], lr_preds[market]
        rows[market] = {
            "GRU %": pooled(g)["accuracy %"],
            "LR %": pooled(l)["accuracy %"],
            "fair bar %": pooled(g)["fair bar %"],
            "p GRU vs fair": pooled(g)["p vs fair"],
            "p GRU vs LR": mcnemar(g["up"], g["p_up"] > 0.5, l["p_up"] > 0.5)[2],
            "GRU AUC": pooled(g)["AUC"],
            "LR AUC": pooled(l)["AUC"],
            "GRU one-answer folds": one_answer_folds(g),
        }
    return pd.DataFrame(rows).T

for task in TASKS:
    print(task)
    display(compare(gru[task], lr[task]).round(3))

own returns, close-to-close


,GRU %,LR %,fair bar %,p GRU vs fair,p GRU vs LR,GRU AUC,LR AUC,GRU one-answer folds
Indonesia,53.603,52.711,52.917,0.237,0.165,0.516,0.491,3.0
Malaysia,47.915,49.624,49.761,0.233,0.282,0.487,0.502,0.0
Philippines,50.410,51.573,51.778,0.355,0.434,0.494,0.495,0.0
Singapore,46.813,50.598,49.402,0.163,0.039,0.466,0.503,0.0
Thailand,51.682,52.437,52.437,0.351,0.379,0.522,0.488,1.0
India,54.496,53.414,54.564,1.000,0.350,0.538,0.522,2.0


own returns + US, close-to-close


,GRU %,LR %,fair bar %,p GRU vs fair,p GRU vs LR,GRU AUC,LR AUC,GRU one-answer folds
Indonesia,59.437,57.927,52.917,0.000,0.168,0.629,0.605,0.0
Malaysia,60.834,59.535,49.761,0.000,0.250,0.642,0.635,0.0
Philippines,60.465,58.960,51.778,0.000,0.204,0.647,0.623,0.0
Singapore,61.155,59.695,49.402,0.000,0.230,0.659,0.627,0.0
Thailand,58.065,55.182,52.437,0.000,0.015,0.619,0.581,0.0
India,60.176,60.108,54.564,0.001,1.000,0.628,0.623,0.0


own returns + US, open-to-close


,GRU %,LR %,fair bar %,p GRU vs fair,p GRU vs LR,GRU AUC,LR AUC,GRU one-answer folds
Indonesia,53.074,49.261,49.027,0.020,0.029,0.542,0.528,0.0
Malaysia,51.849,50.479,48.904,0.044,0.354,0.531,0.532,1.0
Philippines,55.715,51.951,50.171,0.001,0.010,0.579,0.525,0.0
Singapore,46.977,48.571,49.767,0.109,0.328,0.451,0.466,2.0
Thailand,52.780,54.839,54.358,0.396,0.236,0.535,0.523,0.0
India,53.518,51.556,52.503,0.423,0.108,0.530,0.497,0.0


## GRU vs the simplest model

Here **LR** is logistic regression on one input: the last US move, as in step 5. All six test years pooled.

In [4]:
def us_only(df):
    return make_us_features(df, us)

simple = {}
for task in list(TASKS)[1:]:  # the two tasks with US data
    simple[task] = run_all_markets(fit_classifier, CLASSIFIERS["Logistic regression"], pick_by_log_loss, us_only, TASKS[task][1])
    print(task)
    display(compare(gru[task], simple[task]).round(3))

own returns + US, close-to-close


,GRU %,LR %,fair bar %,p GRU vs fair,p GRU vs LR,GRU AUC,LR AUC,GRU one-answer folds
Indonesia,59.437,57.996,52.917,0.000,0.135,0.629,0.621,0.0
Malaysia,60.834,59.945,49.761,0.000,0.322,0.642,0.648,0.0
Philippines,60.465,60.465,51.778,0.000,1.000,0.647,0.644,0.0
Singapore,61.155,61.421,49.402,0.000,0.823,0.659,0.657,0.0
Thailand,58.065,57.790,52.437,0.000,0.834,0.619,0.619,0.0
India,60.176,60.784,54.564,0.001,0.656,0.628,0.637,0.0


own returns + US, open-to-close


,GRU %,LR %,fair bar %,p GRU vs fair,p GRU vs LR,GRU AUC,LR AUC,GRU one-answer folds
Indonesia,53.074,48.327,49.027,0.020,0.007,0.542,0.504,0.0
Malaysia,51.849,49.795,48.904,0.044,0.135,0.531,0.550,1.0
Philippines,55.715,55.921,50.171,0.001,0.904,0.579,0.580,0.0
Singapore,46.977,49.369,49.767,0.109,0.168,0.451,0.478,2.0
Thailand,52.780,54.770,54.358,0.396,0.209,0.535,0.528,0.0
India,53.518,52.436,52.503,0.423,0.385,0.530,0.494,0.0


## All 36 folds

"Significantly" = McNemar p < 0.05 against the fair bar.

In [5]:
everything = {}
for task in TASKS:
    everything[f"{task} | GRU"] = gru[task]
    everything[f"{task} | LR"] = lr[task]
overview(everything).round(3)

,mean AUC,above fair bar,below fair bar,significantly above,significantly below,above stretch bar,one-answer folds
"own returns, close-to-close | GRU",0.497,7.0,19.0,0.0,1.0,5.0,6.0
"own returns, close-to-close | LR",0.501,4.0,12.0,0.0,0.0,2.0,18.0
"own returns + US, close-to-close | GRU",0.639,34.0,2.0,17.0,0.0,34.0,0.0
"own returns + US, close-to-close | LR",0.618,31.0,2.0,13.0,0.0,30.0,2.0
"own returns + US, open-to-close | GRU",0.534,20.0,9.0,2.0,0.0,12.0,3.0
"own returns + US, open-to-close | LR",0.520,12.0,9.0,1.0,0.0,11.0,15.0


## Full tables for the GRU (step 3 style)

In [6]:
for task in TASKS:
    print(task)
    display(model_table(gru[task]).round(3))

own returns, close-to-close


,accuracy %,MCC,AUC,fair bar %,p vs fair,same as yesterday %,p vs same as yesterday,called up %,one-answer folds
Indonesia,53.603,0.047,0.516,52.917,0.237,49.485,0.024,96.019,3.0
Malaysia,47.915,-0.043,0.487,49.761,0.233,49.282,0.405,67.532,0.0
Philippines,50.410,-0.008,0.494,51.778,0.355,48.427,0.298,71.135,0.0
Singapore,46.813,-0.061,0.466,49.402,0.163,49.602,0.120,33.997,0.0
Thailand,51.682,-0.014,0.522,52.437,0.351,50.995,0.712,92.107,1.0
India,54.496,0.040,0.538,54.564,1.000,54.767,0.889,83.029,2.0


own returns + US, close-to-close


,accuracy %,MCC,AUC,fair bar %,p vs fair,same as yesterday %,p vs same as yesterday,called up %,one-answer folds
Indonesia,59.437,0.180,0.629,52.917,0.000,49.485,0.000,67.948,0.0
Malaysia,60.834,0.219,0.642,49.761,0.000,49.282,0.000,56.528,0.0
Philippines,60.465,0.207,0.647,51.778,0.000,48.427,0.000,56.703,0.0
Singapore,61.155,0.223,0.659,49.402,0.000,49.602,0.000,52.324,0.0
Thailand,58.065,0.155,0.619,52.437,0.000,50.995,0.000,70.213,0.0
India,60.176,0.194,0.628,54.564,0.001,54.767,0.001,56.389,0.0


own returns + US, open-to-close


,accuracy %,MCC,AUC,fair bar %,p vs fair,same as yesterday %,p vs same as yesterday,called up %,one-answer folds
Indonesia,53.074,0.069,0.542,49.027,0.020,48.171,0.018,62.646,0.0
Malaysia,51.849,0.050,0.531,48.904,0.044,50.000,0.344,70.068,1.0
Philippines,55.715,0.115,0.579,50.171,0.001,48.597,0.000,58.042,0.0
Singapore,46.977,-0.060,0.451,49.767,0.109,49.302,0.238,43.455,2.0
Thailand,52.780,0.049,0.535,54.358,0.396,47.975,0.017,46.054,0.0
India,53.518,0.051,0.530,52.503,0.423,48.444,0.003,20.636,0.0


## Do the open-to-close edges survive other random seeds?

Open-to-close with own returns + US, for the three markets where seed 0 beat the fair bar. **average day in %** is what a trade earns on the index before costs. About 15 minutes.

In [7]:
SEEDS = [0, 1, 2, 3]
WATCH = ["Indonesia", "Malaysia", "Philippines"]

def one_seed(df, seed):
    return walk_forward(df, partial(fit_gru, seed=seed), HIDDEN_SIZES, make_x=with_us, make_y=up_open_to_close,
                        pick=partial(pick_gru_setting, seed=seed))

keys = [(market, seed) for market in WATCH for seed in SEEDS]
runs = Parallel(n_jobs=6)(delayed(one_seed)(markets[market], seed) for market, seed in keys)

rows = {}
for (market, seed), p in zip(keys, runs):
    df = markets[market].loc[p.index]
    day_return = df["Close"] / df["Open"] - 1
    calls_up = p["p_up"] > 0.5
    scores = pooled(p)
    rows[market, seed] = {
        "GRU %": scores["accuracy %"],
        "fair bar %": scores["fair bar %"],
        "p vs fair": scores["p vs fair"],
        "AUC": scores["AUC"],
        "days in": calls_up.sum(),
        "average day in %": 100 * day_return[calls_up].mean(),
    }
pd.DataFrame(rows).T.round(3)

GRU %  fair bar %  p vs fair    AUC  days in  average day in %
Indonesia   0  53.074      49.027      0.020  0.542    805.0             0.003
            1  52.140      49.027      0.121  0.522    651.0            -0.007
            2  50.117      49.027      0.444  0.539    997.0            -0.039
            3  52.763      49.027      0.048  0.541    721.0            -0.012
Malaysia    0  51.849      48.904      0.044  0.531   1023.0             0.019
            1  49.658      48.904      0.572  0.529   1147.0             0.000
            2  50.068      48.904      0.311  0.530   1211.0             0.004
            3  51.781      48.904      0.063  0.540    972.0             0.034
Philippines 0  55.715      50.171      0.001  0.579    848.0             0.120
            1  56.194      50.171      0.000  0.576    871.0             0.125
            2  56.331      50.171      0.000  0.577    885.0             0.131
            3  55.099      50.171      0.003  0.578    887.0             0.106

## What did tuning pick?

(hidden size, epochs) for each market and test year:

In [8]:
pd.DataFrame({market: p.groupby("year")["setting"].first()
              for market, p in gru["own returns, close-to-close"].items()})

,Indonesia,Malaysia,Philippines,Singapore,Thailand,India
year,,,,,,
2017,"(16, 44)","(8, 130)","(8, 9)","(8, 7)","(32, 23)","(16, 3)"
2018,"(16, 22)","(16, 2)","(16, 2)","(16, 5)","(8, 16)","(32, 23)"
2019,"(32, 11)","(8, 15)","(8, 4)","(8, 8)","(8, 7)","(32, 9)"
2020,"(8, 6)","(8, 3)","(16, 1)","(16, 92)","(8, 5)","(32, 70)"
2021,"(32, 1)","(8, 6)","(16, 94)","(8, 3)","(8, 5)","(16, 3)"
2022,"(16, 3)","(32, 7)","(16, 40)","(16, 1)","(16, 1)","(8, 54)"
